In [1]:
# uni of lucknow

In [2]:
import os
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================

BASE_URL = "https://www.lkouniv.ac.in"

SOURCES = {
    "general": (
        "https://www.lkouniv.ac.in/en/news"
        "?Newslistslug=en-notices&cd=MwAzADcA"
    ),

    "examination": (
        "https://lkouniv.ac.in/en/news"
        "?Newslistslug=en-examination-schedule&cd=OAA1AA%3D%3D"
    ),

    "student": (
        "https://www.lkouniv.ac.in/en/news"
        "?newslistslug=en-student-news"
    ),

    "events": (
        "https://www.lkouniv.ac.in/en/news"
        "?Newslistslug=en-events"
    ),
}


INSTITUTION = "University of Lucknow"

TARGET_DOCUMENTS = 20

OUTPUT_DIR = "data/raw"
PDF_DIR = os.path.join(OUTPUT_DIR, "pdfs")
METADATA_FILE = os.path.join(
    OUTPUT_DIR,
    "metadata.csv"
)

os.makedirs(PDF_DIR, exist_ok=True)


# ============================================================
# HTTP
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/120 Safari/537.36"
    )
}


def get_page(url):

    response = requests.get(
        url,
        timeout=30,
        headers=HEADERS
    )

    response.raise_for_status()

    return response.text


# ============================================================
# CLEAN FILENAME
# ============================================================

def clean_filename(text):

    text = re.sub(
        r"[^\w\s-]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        "_",
        text.strip()
    )

    return text[:100]


# ============================================================
# EXTRACT ENGLISH PDF LINKS
# ============================================================

def extract_notices(
    html,
    category
):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    notices = []

    # Find PDF links
    for link in soup.find_all(
        "a",
        href=True
    ):

        href = link["href"]

        # Only PDFs
        if ".pdf" not in href.lower():
            continue

        pdf_url = urljoin(
            BASE_URL,
            href
        )

        # ----------------------------------------------------
        # Get surrounding text.
        #
        # Lucknow's page contains the subject and language
        # information around the PDF link.
        # ----------------------------------------------------

        parent = link.parent

        surrounding_text = ""

        if parent:
            surrounding_text = parent.get_text(
                " ",
                strip=True
            )

        # Also check a few parent levels upward
        for _ in range(3):

            if parent:

                parent = parent.parent

                if parent:

                    text = parent.get_text(
                        " ",
                        strip=True
                    )

                    if len(text) > len(
                        surrounding_text
                    ):
                        surrounding_text = text

        # ----------------------------------------------------
        # LANGUAGE FILTER
        # ----------------------------------------------------

        language_match = re.search(
            r"Language\s*:\s*(English|Hindi)",
            surrounding_text,
            re.IGNORECASE
        )

        if not language_match:
            continue

        language = (
            language_match
            .group(1)
            .strip()
            .title()
        )

        # We only want English for this dataset
        if language != "English":
            continue

        # ----------------------------------------------------
        # SUBJECT
        # ----------------------------------------------------

        text = link.get_text(
            " ",
            strip=True
        )

        # Sometimes the PDF link itself has no useful text.
        # Use surrounding text as fallback.
        if not text:
            text = surrounding_text

        notices.append({

            "institution": INSTITUTION,

            "category": category,

            "subject": text,

            "language": language,

            "pdf_url": pdf_url,
        })

    return notices


# ============================================================
# DOWNLOAD
# ============================================================

def download_pdf(
    notice,
    index
):

    subject = notice["subject"]

    filename = clean_filename(
        subject
    )

    if not filename:

        filename = (
            f"lucknow_notice_{index}"
        )

    filename = (
        f"{index:04d}_{filename}.pdf"
    )

    filepath = os.path.join(
        PDF_DIR,
        filename
    )

    # Already downloaded
    if os.path.exists(filepath):

        return filepath

    try:

        response = requests.get(
            notice["pdf_url"],
            timeout=60,
            headers=HEADERS
        )

        response.raise_for_status()

        content_type = (
            response
            .headers
            .get("Content-Type", "")
            .lower()
        )

        if "pdf" not in content_type:

            print(
                f"Skipping non-PDF: "
                f"{notice['pdf_url']}"
            )

            return None

        with open(
            filepath,
            "wb"
        ) as f:

            f.write(
                response.content
            )

        return filepath

    except Exception as e:

        print(
            f"Failed: "
            f"{notice['pdf_url']}\n"
            f"Reason: {e}"
        )

        return None


# ============================================================
# MAIN
# ============================================================

def main():

    all_notices = []

    print(
        "\nCollecting English PDFs "
        "from University of Lucknow...\n"
    )

    # --------------------------------------------------------
    # SCRAPE ALL SOURCE CATEGORIES
    # --------------------------------------------------------

    for category, url in SOURCES.items():

        print(
            f"Scraping: {category}"
        )

        try:

            html = get_page(url)

            notices = extract_notices(
                html,
                category
            )

            print(
                f"  English PDFs found: "
                f"{len(notices)}"
            )

            all_notices.extend(
                notices
            )

        except Exception as e:

            print(
                f"  Failed: {e}"
            )

    # --------------------------------------------------------
    # DEDUPLICATE BY PDF URL
    # --------------------------------------------------------

    unique = {}

    for notice in all_notices:

        unique[
            notice["pdf_url"]
        ] = notice

    all_notices = list(
        unique.values()
    )

    print(
        f"\nUnique English PDFs found: "
        f"{len(all_notices)}"
    )

    # --------------------------------------------------------
    # TAKE ONLY TARGET NUMBER
    # --------------------------------------------------------

    selected = all_notices[
        :TARGET_DOCUMENTS
    ]

    print(
        f"Selecting: "
        f"{len(selected)} documents"
    )

    # --------------------------------------------------------
    # DOWNLOAD
    # --------------------------------------------------------

    records = []

    print(
        "\nDownloading...\n"
    )

    for i, notice in enumerate(
        tqdm(selected),
        start=1
    ):

        filepath = download_pdf(
            notice,
            i
        )

        if filepath:

            records.append({

                "id": i,

                "institution": (
                    notice["institution"]
                ),

                "category": (
                    notice["category"]
                ),

                "subject": (
                    notice["subject"]
                ),

                "language": (
                    notice["language"]
                ),

                "pdf_url": (
                    notice["pdf_url"]
                ),

                "local_path": (
                    filepath
                ),
            })

        # Be polite to the server
        time.sleep(0.5)

    # --------------------------------------------------------
    # SAVE METADATA
    # --------------------------------------------------------

    df = pd.DataFrame(
        records
    )

    df.to_csv(
        METADATA_FILE,
        index=False
    )

    print(
        "\n=============================="
    )

    print(
        "DONE"
    )

    print(
        f"Downloaded: {len(df)} PDFs"
    )

    print(
        f"Metadata: {METADATA_FILE}"
    )

    print(
        f"PDF folder: {PDF_DIR}"
    )

    print(
        "=============================="
    )


if __name__ == "__main__":

    main()



Scraping: general
  English PDFs found: 10
Scraping: examination
  English PDFs found: 10
Scraping: student
  English PDFs found: 0
Scraping: events
  English PDFs found: 10

Unique English PDFs found: 30
Selecting: 20 documents

Downloading...



100%|██████████| 20/20 [00:15<00:00,  1.28it/s]


DONE
Downloaded: 20 PDFs
Metadata: data/raw/metadata.csv
PDF folder: data/raw/pdfs


In [7]:
# =========================
# ADD 20 JNU PDFs (IDs 21-40)
# =========================

import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm.auto import tqdm

BASE_URL = "https://www.jnu.ac.in"
SOURCE_URL = "https://www.jnu.ac.in/index.php/notices"

METADATA_FILE = "data/raw/metadata.csv"
PDF_DIR = "data/raw/pdfs"

os.makedirs(PDF_DIR, exist_ok=True)


# ---------- Fetch page ----------
html = requests.get(
    SOURCE_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
).text

soup = BeautifulSoup(html, "html.parser")


# ---------- Find usable JNU notice PDFs ----------
skip_titles = [
    "prospectus",
    "programme list",
    "admission policy",
    "honorary degrees",
    "finance committee",
    "executive council",
    "academic council",
    "university court",
]

notice_keywords = [
    "circular",
    "notice",
    "extension",
    "commencement",
    "registration",
    "notification",
    "public notice",
    "statement",
    "advisory",
    "holiday",
    "add/drop",
    "office order",
    "examination",
    "admission",
    "semester",
]

skip_urls = {
    "https://www.jnu.ac.in/sites/default/files/Court.pdf",
    "https://www.jnu.ac.in/sites/default/files/EC.pdf",
    "https://www.jnu.ac.in/sites/default/files/AC.pdf",
    "https://www.jnu.ac.in/sites/default/files/FC.pdf",
}

notices = []
seen_urls = set()

for link in soup.find_all("a", href=True):

    href = link["href"]

    if ".pdf" not in href.lower():
        continue

    pdf_url = urljoin(BASE_URL, href)

    if pdf_url in seen_urls or pdf_url in skip_urls:
        continue

    seen_urls.add(pdf_url)

    title = link.get_text(" ", strip=True)

    if not title or title.lower() == "download":
        continue

    lower_title = title.lower()
    lower_url = pdf_url.lower()

    # Skip obvious non-notice documents
    if any(x in lower_title for x in skip_titles):
        continue

    if "prospectus" in lower_url:
        continue

    # Collection hygiene only
    if not any(k in lower_title for k in notice_keywords):
        continue

    notices.append({
        "institution": "Jawaharlal Nehru University",
        "category": "general",
        "subject": title,
        "language": "English",
        "pdf_url": pdf_url
    })

    if len(notices) == 20:
        break


print(f"JNU notices selected: {len(notices)}")

for i, n in enumerate(notices, 21):
    print(f"{i}. {n['subject']}")


# ---------- Download + append metadata ----------
records = []

for doc_id, notice in tqdm(
    zip(range(21, 41), notices),
    total=len(notices)
):

    filename = f"{doc_id:04d}.pdf"
    filepath = os.path.join(PDF_DIR, filename)

    try:
        response = requests.get(
            notice["pdf_url"],
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=60
        )
        response.raise_for_status()

        with open(filepath, "wb") as f:
            f.write(response.content)

        records.append({
            "id": doc_id,
            "institution": notice["institution"],
            "category": notice["category"],
            "subject": notice["subject"],
            "language": notice["language"],
            "pdf_url": notice["pdf_url"],
            "local_path": filepath
        })

    except Exception as e:
        print(f"\nFAILED {doc_id}: {notice['subject']}")
        print(e)

    time.sleep(0.5)


# ---------- Update metadata.csv ----------
new_df = pd.DataFrame(records)

if os.path.exists(METADATA_FILE):
    existing_df = pd.read_csv(METADATA_FILE)
    final_df = pd.concat(
        [existing_df, new_df],
        ignore_index=True
    )
else:
    final_df = new_df

final_df.to_csv(METADATA_FILE, index=False)

print("\n" + "=" * 40)
print("DONE")
print(f"Added JNU PDFs: {len(new_df)}")
print(f"Total dataset: {len(final_df)} PDFs")
print(f"Metadata: {METADATA_FILE}")
print("=" * 40)

JNU notices selected: 11
21. Circular regarding holiday on 11.09.2026
22. Extension of date for Add/Drop courses for newly admitted students for Monsoon Semester 2026
23. Commencement of classes for newly admitted students of the UG Prog (AS 2026-27)
24. Extension of registration and Add/Drop date for Monsoon Semester 2026
25. Registration for Monsoon Semester 2026
26. Public Notice dt: 19-06-2026 reg. Section 49
27. Circular reg. change in date of holiday on account of Eid-ul-Zuha (Bakrid)
28. Circular regarding Holiday on 14th April, 2026
29. Statement dt: 23-02-2026
30. Notification for commencement of classes for Ph.D Winter 2026 batch
31. Extension of Add/Drop date for Winter Semester 2026


  0%|          | 0/11 [00:00<?, ?it/s]


DONE
Added JNU PDFs: 11
Total dataset: 31 PDFs
Metadata: data/raw/metadata.csv


In [9]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm.auto import tqdm

# =========================
# JNU: COLLECT EVERYTHING AVAILABLE
# =========================

BASE_URL = "https://www.jnu.ac.in"

SOURCE_URLS = [
    "https://www.jnu.ac.in/index.php/notices",
    "https://www.jnu.ac.in/index.php/iha-notices",
]

METADATA_FILE = "data/raw/metadata.csv"
PDF_DIR = "data/raw/pdfs"

os.makedirs(PDF_DIR, exist_ok=True)


# ---------- Existing metadata ----------
df = pd.read_csv(METADATA_FILE)

existing_urls = set(
    df["pdf_url"]
    .dropna()
    .astype(str)
)

next_id = int(df["id"].max()) + 1

print(f"Existing documents: {len(df)}")
print(f"Next ID: {next_id}")


# ---------- Things we definitely DON'T want ----------
skip_titles = [
    "prospectus",
    "programme list",
    "admission policy",
    "honorary degrees",
    "finance committee",
    "executive council",
    "academic council",
    "university court",
]

skip_url_parts = [
    "prospectus",
]


# ---------- Collect PDFs ----------
all_new = []
seen_urls = set()

for source_url in SOURCE_URLS:

    print(f"\nScanning: {source_url}")

    response = requests.get(
        source_url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    source_count = 0

    for link in soup.find_all("a", href=True):

        href = link["href"]

        if ".pdf" not in href.lower():
            continue

        pdf_url = urljoin(BASE_URL, href)

        # Already in dataset
        if pdf_url in existing_urls:
            continue

        # Duplicate within this run
        if pdf_url in seen_urls:
            continue

        seen_urls.add(pdf_url)

        title = link.get_text(" ", strip=True)

        if not title:
            continue

        lower_title = title.lower()
        lower_url = pdf_url.lower()

        # Skip obvious non-document links
        if lower_title in [
            "download",
            "click here",
            "view",
        ]:
            continue

        # Skip obvious prospectus/governance docs
        if any(x in lower_title for x in skip_titles):
            continue

        if any(x in lower_url for x in skip_url_parts):
            continue

        # Add document
        all_new.append({
            "institution": "Jawaharlal Nehru University",
            "category": (
                "hostel"
                if "iha-notices" in source_url
                else "general"
            ),
            "subject": title,
            "language": "English",
            "pdf_url": pdf_url
        })

        source_count += 1

    print(f"New PDFs found from source: {source_count}")


# ---------- Show everything before downloading ----------
print("\n" + "=" * 60)
print(f"TOTAL NEW JNU PDFs FOUND: {len(all_new)}")
print("=" * 60)

for i, doc in enumerate(all_new, start=next_id):
    print(f"{i}. {doc['subject']}")


# ---------- Download ----------
records = []

for doc_id, doc in tqdm(
    zip(range(next_id, next_id + len(all_new)), all_new),
    total=len(all_new)
):

    filepath = os.path.join(
        PDF_DIR,
        f"{doc_id:04d}.pdf"
    )

    try:

        r = requests.get(
            doc["pdf_url"],
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=60
        )

        r.raise_for_status()

        # Basic PDF validation
        if not r.content.startswith(b"%PDF"):
            print(f"\nWARNING: Not a normal PDF: {doc['subject']}")
            continue

        with open(filepath, "wb") as f:
            f.write(r.content)

        records.append({
            "id": doc_id,
            "institution": doc["institution"],
            "category": doc["category"],
            "subject": doc["subject"],
            "language": doc["language"],
            "pdf_url": doc["pdf_url"],
            "local_path": filepath
        })

    except Exception as e:

        print(
            f"\nFAILED {doc_id}: "
            f"{doc['subject']}\n{e}"
        )

    time.sleep(0.3)


# ---------- Update metadata ----------
if records:

    new_df = pd.DataFrame(records)

    final_df = pd.concat(
        [df, new_df],
        ignore_index=True
    )

    final_df.to_csv(
        METADATA_FILE,
        index=False
    )

else:

    final_df = df


# ---------- Final summary ----------
print("\n" + "=" * 60)
print("JNU COLLECTION COMPLETE")
print("=" * 60)
print(f"Previously present: {len(df)}")
print(f"Newly added:        {len(records)}")
print(f"Total dataset:      {len(final_df)}")
print(f"Metadata:            {METADATA_FILE}")
print(f"PDF directory:       {PDF_DIR}")
print("=" * 60)

Existing documents: 35
Next ID: 36

Scanning: https://www.jnu.ac.in/index.php/notices
New PDFs found from source: 0

Scanning: https://www.jnu.ac.in/index.php/iha-notices
New PDFs found from source: 11

TOTAL NEW JNU PDFs FOUND: 11
36. Manual
37. Tentative seniority list for single seater boys 2026-.pdf
38. Tentative seniority list for single seater boys 2026......pdf
39. Tentative seniority list for single seater girls  2026.pdf
40. PG Boys dormitory allotment list 2026-27...pdf
41. PG Girls Dormitory allotment.pdf
42. Re-revised schedule for single seater 2026-27-1.pdf
43. UG BOYS DORMITORY ALLOTMENT LIST 2026-27 (2).pdf
44. UG BOYS B-TECH. STUDENTS HOSTEL ALLOTMENT 2026-27.pdf
45. PG GIRLS DORMITORY ALLOTMENT LIST.pdf
46. PG boys hostel allotment list 2026-27.pdf


  0%|          | 0/11 [00:00<?, ?it/s]


JNU COLLECTION COMPLETE
Previously present: 35
Newly added:        11
Total dataset:      46
Metadata:            data/raw/metadata.csv
PDF directory:       data/raw/pdfs


In [10]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BHU_URLS = [
    "https://www.bhu.ac.in/Site/Home/1_2_16_Main-Site",
    "https://admission.bhu.ac.in/en",
]

for url in BHU_URLS:
    print("\n" + "=" * 70)
    print("SCANNING:", url)

    r = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )

    print("Status:", r.status_code)
    print("Final URL:", r.url)
    print("HTML length:", len(r.text))

    soup = BeautifulSoup(r.text, "html.parser")

    pdfs = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if ".pdf" in href.lower():
            pdf_url = urljoin(r.url, href)
            title = a.get_text(" ", strip=True)

            pdfs.append((title, pdf_url))

    print("PDF links found:", len(pdfs))

    for i, (title, pdf_url) in enumerate(pdfs[:30], 1):
        print(f"{i}. {title}")
        print(f"   {pdf_url}")


SCANNING: https://www.bhu.ac.in/Site/Home/1_2_16_Main-Site
Status: 200
Final URL: https://www.bhu.ac.in/Site/Home/1_2_16_Main-Site
HTML length: 48020
PDF links found: 0

SCANNING: https://admission.bhu.ac.in/en
Status: 200
Final URL: https://admission.bhu.ac.in/en
HTML length: 636009
PDF links found: 77
1. Information Bulletin
   https://storage.bhu.ac.in/strapi-media/uploads/FINAL_DRAFT_BHU_UG_2026_Information_Bulletin_23032026_b86ed3fbb0.pdf
2. Information Bulletin
   https://storage.bhu.ac.in/strapi-media/uploads/PG_Bulletin_2026_Final_compressed_51f86ccb53.pdf
3. Information Bulletin
   https://storage.bhu.ac.in/strapi-media/uploads/Information_Bulletin_BVSC_AH_31082026_dc82e37786.pdf
4. 
   https://storage.bhu.ac.in/strapi-media/uploads/UG_Mop_Up_R2_8aa2b0a4e4.pdf
5. 
   https://storage.bhu.ac.in/strapi-media/uploads/Ug_mop_up_reporting_2026_39d4e4ef92.pdf
6. 
   https://storage.bhu.ac.in/strapi-media/uploads/BPA_BFA_round_1_16092026_1bba57588f.pdf
7. 
   https://storage.bhu.ac.i

In [11]:
# BHU
# =========================
# BHU ADMISSION PDF COLLECTOR
# =========================

import os
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse, unquote
from tqdm.auto import tqdm

SOURCE_URL = "https://admission.bhu.ac.in/en"

METADATA_FILE = "data/raw/metadata.csv"
PDF_DIR = "data/raw/pdfs"

os.makedirs(PDF_DIR, exist_ok=True)


# ---------- Load existing dataset ----------
df = pd.read_csv(METADATA_FILE)

existing_urls = set(
    df["pdf_url"].dropna().astype(str)
)

next_id = int(df["id"].max()) + 1

print("Existing documents:", len(df))
print("Next ID:", next_id)


# ---------- Fetch BHU page ----------
response = requests.get(
    SOURCE_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


# ---------- Extract PDFs ----------
documents = []
seen_urls = set()

for link in soup.find_all("a", href=True):

    href = link["href"]

    if ".pdf" not in href.lower():
        continue

    pdf_url = href.strip()

    # Already collected
    if pdf_url in existing_urls:
        continue

    # Duplicate on page
    if pdf_url in seen_urls:
        continue

    seen_urls.add(pdf_url)

    # Get title
    title = link.get_text(" ", strip=True)

    # If title is blank, use filename
    if not title:
        filename = os.path.basename(
            urlparse(pdf_url).path
        )

        title = unquote(filename)

        # Remove random hash before .pdf if useful
        title = re.sub(
            r"_[a-f0-9]{8,}(?=\.pdf$)",
            "",
            title,
            flags=re.IGNORECASE
        )

    documents.append({
        "institution": "Banaras Hindu University",
        "category": "admission",
        "subject": title,
        "language": "English",
        "pdf_url": pdf_url
    })


print("\nNew BHU PDFs found:", len(documents))

for i, doc in enumerate(documents, start=next_id):
    print(f"{i}. {doc['subject']}")
    print(f"   {doc['pdf_url']}")


# ---------- Download ----------
records = []

for doc_id, doc in tqdm(
    zip(
        range(next_id, next_id + len(documents)),
        documents
    ),
    total=len(documents)
):

    # Safe filename
    filepath = os.path.join(
        PDF_DIR,
        f"{doc_id:04d}.pdf"
    )

    try:

        r = requests.get(
            doc["pdf_url"],
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=90
        )

        r.raise_for_status()

        # Make sure it's actually a PDF
        if not r.content.startswith(b"%PDF"):
            print(
                f"\nWARNING: Not a valid PDF: "
                f"{doc['subject']}"
            )
            continue

        with open(filepath, "wb") as f:
            f.write(r.content)

        records.append({
            "id": doc_id,
            "institution": doc["institution"],
            "category": doc["category"],
            "subject": doc["subject"],
            "language": doc["language"],
            "pdf_url": doc["pdf_url"],
            "local_path": filepath
        })

    except Exception as e:

        print(
            f"\nFAILED {doc_id}: "
            f"{doc['subject']}\n{e}"
        )

    time.sleep(0.3)


# ---------- Update metadata ----------
if records:

    new_df = pd.DataFrame(records)

    final_df = pd.concat(
        [df, new_df],
        ignore_index=True
    )

    final_df.to_csv(
        METADATA_FILE,
        index=False
    )

else:
    final_df = df


# ---------- Summary ----------
print("\n" + "=" * 60)
print("BHU COLLECTION COMPLETE")
print("=" * 60)
print("Previously present:", len(df))
print("Newly added:", len(records))
print("Total dataset:", len(final_df))
print("Metadata:", METADATA_FILE)
print("=" * 60)

Existing documents: 46
Next ID: 47

New BHU PDFs found: 76
47. Information Bulletin
   https://storage.bhu.ac.in/strapi-media/uploads/FINAL_DRAFT_BHU_UG_2026_Information_Bulletin_23032026_b86ed3fbb0.pdf
48. Information Bulletin
   https://storage.bhu.ac.in/strapi-media/uploads/PG_Bulletin_2026_Final_compressed_51f86ccb53.pdf
49. Information Bulletin
   https://storage.bhu.ac.in/strapi-media/uploads/Information_Bulletin_BVSC_AH_31082026_dc82e37786.pdf
50. UG_Mop_Up_R2.pdf
   https://storage.bhu.ac.in/strapi-media/uploads/UG_Mop_Up_R2_8aa2b0a4e4.pdf
51. Ug_mop_up_reporting_2026.pdf
   https://storage.bhu.ac.in/strapi-media/uploads/Ug_mop_up_reporting_2026_39d4e4ef92.pdf
52. BPA_BFA_round_1_16092026.pdf
   https://storage.bhu.ac.in/strapi-media/uploads/BPA_BFA_round_1_16092026_1bba57588f.pdf
53. UG_Mop_Up_R1.pdf
   https://storage.bhu.ac.in/strapi-media/uploads/UG_Mop_Up_R1_97837becef.pdf
54. BHU_UG_MOP_UP_vacancy_list.pdf
   https://storage.bhu.ac.in/strapi-media/uploads/BHU_UG_MOP_UP_va

  0%|          | 0/76 [00:00<?, ?it/s]


BHU COLLECTION COMPLETE
Previously present: 46
Newly added: 76
Total dataset: 122
Metadata: data/raw/metadata.csv


In [12]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

IITB_URLS = [
    "https://www.iitb.ac.in/announcements",
    "https://acad.iitb.ac.in/circulars",
    "https://acad.iitb.ac.in/",
]

for url in IITB_URLS:

    print("\n" + "=" * 70)
    print("SCANNING:", url)

    r = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )

    print("Status:", r.status_code)
    print("Final URL:", r.url)
    print("HTML length:", len(r.text))

    soup = BeautifulSoup(r.text, "html.parser")

    pdfs = []

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if ".pdf" not in href.lower():
            continue

        pdf_url = urljoin(r.url, href)
        title = a.get_text(" ", strip=True)

        pdfs.append((title, pdf_url))

    print("PDF links found:", len(pdfs))

    for i, (title, pdf_url) in enumerate(pdfs[:40], 1):

        print(f"{i}. {title}")
        print(f"   {pdf_url}")


SCANNING: https://www.iitb.ac.in/announcements
Status: 200
Final URL: https://www.iitb.ac.in/announcements
HTML length: 137690
PDF links found: 4
1. Code of Conduct
   https://www.iitb.ac.in/sites/default/files/2025-11/campusCodeOfConduct.pdf
2. Code of Conduct
   https://www.iitb.ac.in/sites/www.iitb.ac.in/files/2025-11/campusCodeOfConduct.pdf
3. Code of Conduct
   https://www.iitb.ac.in/sites/www.iitb.ac.in/files/2025-11/campusCodeOfConduct.pdf
4. Second 4-Day In-Person Administrative Training Program at IIT Bombay; November 1 to 4, 2025 (last date for receiving nominations is extended till August 18,2025)
   https://www.iitb.ac.in/sites/www.iitb.ac.in/files/2025-06/Second%20Administrative%20Training%20Program%20IITB.pdf 

SCANNING: https://acad.iitb.ac.in/circulars
Status: 200
Final URL: https://acad.iitb.ac.in/circulars
HTML length: 32653
PDF links found: 5
1. Office Memorandum for Scholarships/ Remission of fees UG & M.Sc. students (2026 Batch)
   https://acad.iitb.ac.in/sites/de

In [13]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm.auto import tqdm

# =========================
# IIT BOMBAY CURRENT SOURCES
# =========================

IITB_URLS = [
    "https://www.iitb.ac.in/announcements",
    "https://acad.iitb.ac.in/circulars",
]

METADATA_FILE = "data/raw/metadata.csv"
PDF_DIR = "data/raw/pdfs"

os.makedirs(PDF_DIR, exist_ok=True)


# ---------- Existing metadata ----------
df = pd.read_csv(METADATA_FILE)

existing_urls = set(
    df["pdf_url"].dropna().astype(str)
)

next_id = int(df["id"].max()) + 1

print("Existing documents:", len(df))
print("Next ID:", next_id)


# ---------- Collect PDFs ----------
documents = []
seen_normalized = set()

for source_url in IITB_URLS:

    print("\nScanning:", source_url)

    r = requests.get(
        source_url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=60
    )

    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    for link in soup.find_all("a", href=True):

        href = link["href"]

        if ".pdf" not in href.lower():
            continue

        pdf_url = urljoin(r.url, href)

        # Normalize duplicate IITB URLs
        normalized_url = pdf_url.replace(
            "https://www.iitb.ac.in/sites/www.iitb.ac.in/",
            "https://www.iitb.ac.in/sites/"
        )

        # Already in metadata
        if (
            pdf_url in existing_urls
            or normalized_url in existing_urls
        ):
            continue

        # Already discovered
        if normalized_url in seen_normalized:
            continue

        seen_normalized.add(normalized_url)

        title = link.get_text(" ", strip=True)

        if not title:
            title = os.path.basename(
                urlparse(pdf_url).path
            )

        documents.append({
            "institution": "Indian Institute of Technology Bombay",
            "category": (
                "announcements"
                if "announcements" in source_url
                else "academic"
            ),
            "subject": title,
            "language": "English",
            "pdf_url": normalized_url
        })


# ---------- Show collection ----------
print("\n" + "=" * 60)
print("NEW IIT BOMBAY PDFs:", len(documents))
print("=" * 60)

for i, doc in enumerate(documents, start=next_id):
    print(f"{i}. {doc['subject']}")
    print(f"   {doc['pdf_url']}")


# ---------- Download ----------
records = []

for doc_id, doc in tqdm(
    zip(
        range(next_id, next_id + len(documents)),
        documents
    ),
    total=len(documents)
):

    filepath = os.path.join(
        PDF_DIR,
        f"{doc_id:04d}.pdf"
    )

    try:

        r = requests.get(
            doc["pdf_url"],
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=90
        )

        r.raise_for_status()

        if not r.content.startswith(b"%PDF"):
            print(
                f"\nWARNING: Not a PDF: "
                f"{doc['subject']}"
            )
            continue

        with open(filepath, "wb") as f:
            f.write(r.content)

        records.append({
            "id": doc_id,
            "institution": doc["institution"],
            "category": doc["category"],
            "subject": doc["subject"],
            "language": doc["language"],
            "pdf_url": doc["pdf_url"],
            "local_path": filepath
        })

    except Exception as e:

        print(
            f"\nFAILED {doc_id}: "
            f"{doc['subject']}\n{e}"
        )

    time.sleep(0.3)


# ---------- Update metadata ----------
if records:

    new_df = pd.DataFrame(records)

    final_df = pd.concat(
        [df, new_df],
        ignore_index=True
    )

    final_df.to_csv(
        METADATA_FILE,
        index=False
    )

else:
    final_df = df


print("\n" + "=" * 60)
print("IIT BOMBAY COLLECTION COMPLETE")
print("=" * 60)
print("Previously present:", len(df))
print("Newly added:", len(records))
print("Total dataset:", len(final_df))
print("=" * 60)

Existing documents: 122
Next ID: 123

Scanning: https://www.iitb.ac.in/announcements

Scanning: https://acad.iitb.ac.in/circulars

NEW IIT BOMBAY PDFs: 8
123. Code of Conduct
   https://www.iitb.ac.in/sites/default/files/2025-11/campusCodeOfConduct.pdf
124. Code of Conduct
   https://www.iitb.ac.in/sites/files/2025-11/campusCodeOfConduct.pdf
125. Second 4-Day In-Person Administrative Training Program at IIT Bombay; November 1 to 4, 2025 (last date for receiving nominations is extended till August 18,2025)
   https://www.iitb.ac.in/sites/files/2025-06/Second%20Administrative%20Training%20Program%20IITB.pdf 
126. Office Memorandum for Scholarships/ Remission of fees UG & M.Sc. students (2026 Batch)
   https://acad.iitb.ac.in/sites/default/files/Remission_of_fees_UGandM.Sc_._students_2026 Batch.pdf
127. Office Memorandum for Scholarships/ Remission of fees UG & M.Sc. students (Phase B)
   https://acad.iitb.ac.in/sites/default/files/Remission_of_fees_UGandM.Sc_._students_2026 Batch_PhaseB.

  0%|          | 0/8 [00:00<?, ?it/s]


FAILED 124: Code of Conduct
404 Client Error: Not Found for url: https://www.iitb.ac.in/sites/files/2025-11/campusCodeOfConduct.pdf

FAILED 125: Second 4-Day In-Person Administrative Training Program at IIT Bombay; November 1 to 4, 2025 (last date for receiving nominations is extended till August 18,2025)
404 Client Error: Not Found for url: https://www.iitb.ac.in/sites/files/2025-06/Second%20Administrative%20Training%20Program%20IITB.pdf%20

IIT BOMBAY COLLECTION COMPLETE
Previously present: 122
Newly added: 6
Total dataset: 128


In [16]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

DU_URLS = [
    "https://www.du.ac.in/index.php?page=notice-board-view-all",
    "https://www.du.ac.in/index.php?page=notifications",
    "https://www.du.ac.in/index.php?page=guidelines-and-notifications",
    "https://www.du.ac.in/index.php?page=news",
]

for url in DU_URLS:

    print("\n" + "=" * 70)
    print("SCANNING:", url)

    r = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )

    print("Status:", r.status_code)
    print("Final URL:", r.url)
    print("HTML length:", len(r.text))

    soup = BeautifulSoup(r.text, "html.parser")

    pdfs = []
    seen = set()

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if ".pdf" not in href.lower():
            continue

        pdf_url = urljoin(r.url, href)

        if pdf_url in seen:
            continue

        seen.add(pdf_url)

        title = a.get_text(" ", strip=True)

        pdfs.append((title, pdf_url))

    print("PDF links found:", len(pdfs))

    for i, (title, pdf_url) in enumerate(pdfs[:40], 1):
        print(f"{i}. {title}")
        print(f"   {pdf_url}")


SCANNING: https://www.du.ac.in/index.php?page=notice-board-view-all
Status: 200
Final URL: https://www.du.ac.in/index.php?page=notice-board-view-all
HTML length: 709622
PDF links found: 13
1. PMRF Guidelines
   http://du.ac.in/uploads/new-web/09082021_PMRF-guidelines.pdf
2. SC/ST Student Grievance
   http://www.du.ac.in/du/uploads/sc_st_Grievance_1.pdf
3. UGC Guidelines for Student Grievances Regulations, 2023
   https://www.ugc.gov.in/pdfnews/4675881_Regulation.pdf
4. Office Memorandum - 24x7 helpline for women in distress
   https://www.du.ac.in/uploads/new-web/28102021_U5_Section-OM_dated-8102021.pdf
5. Sanctioned Posts
   https://www.du.ac.in/uploads/16032018_Sanctioned%20posts.pdf
6. Welfare Measure
   https://www.du.ac.in/uploads/16032018_Welfare%20Measure_1.pdf
7. Notification - Employee Identity Card
   https://estate.du.ac.in/pdf2023/21.02.2023-notification.pdf
8. "DIGITAL SANSAD" Mobile App
   https://www.du.ac.in/uploads/new-web/17022022_Digital-sansad.pdf
9. Start-Ups Incu

In [18]:
# =========================
# UNIVERSITY OF DELHI
# COLLECT MAX 25 PDFs
# =========================

import os
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, unquote
from tqdm.auto import tqdm


# =========================
# CONFIG
# =========================

DU_URLS = [
    "https://www.du.ac.in/index.php?page=notifications",
    "https://www.du.ac.in/index.php?page=guidelines-and-notifications",
]

MAX_DU_DOCS = 25

METADATA_FILE = "data/raw/metadata.csv"
PDF_DIR = "data/raw/pdfs"

os.makedirs(PDF_DIR, exist_ok=True)


# =========================
# LOAD EXISTING METADATA
# =========================

df = pd.read_csv(METADATA_FILE)

existing_urls = set(
    df["pdf_url"]
    .dropna()
    .astype(str)
)

next_id = int(df["id"].max()) + 1

print("Existing documents:", len(df))
print("Next ID:", next_id)


# =========================
# DU HOST CHECK
# =========================

def is_du_hosted(url):

    host = urlparse(url).netloc.lower()

    return (
        host == "du.ac.in"
        or host.endswith(".du.ac.in")
    )


# =========================
# COLLECT PDF LINKS
# =========================

documents = []
seen_urls = set()


for source_url in DU_URLS:

    print("\nScanning:", source_url)

    r = requests.get(
        source_url,
        headers={
            "User-Agent": "Mozilla/5.0"
        },
        timeout=60
    )

    r.raise_for_status()

    soup = BeautifulSoup(
        r.text,
        "html.parser"
    )

    source_count = 0

    for link in soup.find_all("a", href=True):

        href = link["href"]

        # Only PDF links
        if ".pdf" not in href.lower():
            continue

        # Convert relative URL → absolute URL
        pdf_url = urljoin(
            r.url,
            href
        )

        # Keep only DU-hosted PDFs
        if not is_du_hosted(pdf_url):
            continue

        # Skip already collected PDFs
        if pdf_url in existing_urls:
            continue

        # Skip duplicates between DU pages
        if pdf_url in seen_urls:
            continue

        seen_urls.add(pdf_url)

        # Get title from anchor text
        title = link.get_text(
            " ",
            strip=True
        )

        # Some DU links have blank anchor text
        if not title:

            filename = os.path.basename(
                urlparse(pdf_url).path
            )

            title = unquote(filename)

        documents.append({
            "institution": "University of Delhi",
            "category": (
                "notifications"
                if (
                    "notifications" in source_url
                    and "guidelines" not in source_url
                )
                else "guidelines"
            ),
            "subject": title,
            "language": "English",
            "pdf_url": pdf_url
        })

        source_count += 1

    print(
        "New PDFs from source:",
        source_count
    )


# =========================
# PRIORITIZE RECENT DOCUMENTS
# =========================

def score_du_doc(doc):

    text = (
        doc["subject"]
        + " "
        + doc["pdf_url"]
    ).lower()

    score = 0

    # Prefer recent years
    year_scores = {
        "2026": 10,
        "2025": 9,
        "2024": 8,
        "2023": 7,
        "2022": 6,
        "2021": 5,
        "2020": 4,
        "2019": 3,
        "2018": 2,
        "2017": 1,
    }

    for year, points in year_scores.items():

        if year in text:

            score = max(
                score,
                points
            )

    return score


# Sort by recency
documents = sorted(
    documents,
    key=score_du_doc,
    reverse=True
)


# =========================
# CAP AT 25 DOCUMENTS
# =========================

documents = documents[
    :MAX_DU_DOCS
]


print("\n" + "=" * 60)
print("TOTAL NEW DU PDFs FOUND:", len(seen_urls))
print("DU PDFs SELECTED:", len(documents))
print("=" * 60)


# =========================
# SHOW SELECTED DOCUMENTS
# =========================

for i, doc in enumerate(
    documents,
    start=next_id
):

    print(
        f"{i}. {doc['subject']}"
    )

    print(
        f"   {doc['pdf_url']}"
    )


# =========================
# DOWNLOAD PDFs
# =========================

records = []


for doc_id, doc in tqdm(
    zip(
        range(
            next_id,
            next_id + len(documents)
        ),
        documents
    ),
    total=len(documents)
):

    filepath = os.path.join(
        PDF_DIR,
        f"{doc_id:04d}.pdf"
    )

    try:

        r = requests.get(
            doc["pdf_url"],
            headers={
                "User-Agent": "Mozilla/5.0"
            },
            timeout=90
        )

        r.raise_for_status()

        # Verify that response is actually a PDF
        if not r.content.startswith(
            b"%PDF"
        ):

            print(
                f"\nWARNING: Not a PDF: "
                f"{doc['subject']}"
            )

            continue

        # Save PDF
        with open(
            filepath,
            "wb"
        ) as f:

            f.write(
                r.content
            )

        records.append({

            "id": doc_id,

            "institution":
                doc["institution"],

            "category":
                doc["category"],

            "subject":
                doc["subject"],

            "language":
                doc["language"],

            "pdf_url":
                doc["pdf_url"],

            "local_path":
                filepath
        })

    except Exception as e:

        print(
            f"\nFAILED {doc_id}: "
            f"{doc['subject']}"
        )

        print(e)

    # Small delay between requests
    time.sleep(0.25)


# =========================
# UPDATE METADATA
# =========================

if records:

    new_df = pd.DataFrame(
        records
    )

    final_df = pd.concat(
        [
            df,
            new_df
        ],
        ignore_index=True
    )

    final_df.to_csv(
        METADATA_FILE,
        index=False
    )

else:

    final_df = df


# =========================
# FINAL SUMMARY
# =========================

print("\n" + "=" * 60)
print("DU COLLECTION COMPLETE")
print("=" * 60)

print(
    "Previously present:",
    len(df)
)

print(
    "DU PDFs selected:",
    len(documents)
)

print(
    "Successfully downloaded:",
    len(records)
)

print(
    "Total dataset:",
    len(final_df)
)

print("=" * 60)

Existing documents: 128
Next ID: 131

Scanning: https://www.du.ac.in/index.php?page=notifications
New PDFs from source: 266

Scanning: https://www.du.ac.in/index.php?page=guidelines-and-notifications
New PDFs from source: 90

TOTAL NEW DU PDFs FOUND: 356
DU PDFs SELECTED: 25
131. Promotion Orders No. 26 of 29.03.2020 JACT to Assistants
   https://www.du.ac.in/uploads/Promotion%20Orders%20No.%2026%20of%2029.03.2020%20JACT%20to%20Assistants.pdf
132. Notification regarding Group Insurance Scheme of LIC of India for the Employees of the DU
   https://www.du.ac.in/uploads/2026/11062026-Group-insurance-notice.pdf
133. Associate Professor
   https://www.du.ac.in/uploads/2026/02092026_Post-Based_ReservationRoster-for-AssociateProfessor-updated-upto-30.06.2026.pdf
134. Professor
   https://www.du.ac.in/uploads/2026/01092026_Post-BasedReservationRoster-for-the-Post-of-Professor-updated-upto-30062026.pdf
135. Reservation Roster for PwBD for the post of Assistant Professor updated up to 31.05.2026

  0%|          | 0/25 [00:00<?, ?it/s]

KeyboardInterrupt: 